# Train/Inference Gap Analysis

Visualises results from `gap_experiment.py`.

**State definitions:**
- **A** = post-warmup
- **B** = post-second-phase (clamped for rules 1/4, free for rules 2/3)
- **C** = post-third-phase — what gets stabilised
- **D** = inference endpoint (warmup + free, no clamping)

**The key metric is CD overlap**: if C ≈ D the gap is closed.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from collections import defaultdict

RESULTS_FILE = "../results/single_pattern/gap_results.json"

data = json.loads(Path(RESULTS_FILE).read_text())
results = data["results"]
cfg = data["config"]

RULES = ["current", "approximate", "ep", "long_warmup"]
RULE_LABELS = {
    "current":     "Rule 1: current\n(warmup→clamped→free, stabilise free)",
    "approximate": "Rule 2: approximate\n(warmup→free→clamped, stabilise clamped)",
    "ep":          "Rule 3: EP-like\n(backward(C) − backward(B))",
    "long_warmup": "Rule 4: long warmup\n(long warmup→clamped→free)",
}
COLORS = {"current": "#2196F3", "approximate": "#4CAF50", "ep": "#FF5722", "long_warmup": "#9C27B0"}

def get_traj(rule, key, layer="fc"):
    """Return array (n_seeds, n_steps) for a given metric."""
    runs = [r for r in results if r["rule"] == rule]
    if not runs:
        return None
    col = f"overlaps_{layer}"
    if key in ("soft_margin_C", "accuracy_D"):
        arrs = [[step[key] for step in r["trajectory"]] for r in runs]
    else:
        arrs = [[step[col][key] for step in r["trajectory"]] for r in runs]
    return np.array(arrs)

print(f"Loaded {len(results)} runs | rules: {set(r['rule'] for r in results)}")
print(f"Config: threshold={cfg['threshold']}  lr={cfg['learning_rate']}  "
      f"n_warmup={cfg['n_warmup']}  n_clamped={cfg['n_clamped']}  n_free={cfg['n_free']}")

## Figure 1 — CD overlap and soft margin over updates (FC layer)

The two most important quantities per rule: how quickly the C↔D gap closes and whether C
is actually a good attractor (positive margin).

In [ ]:
fig, axes = plt.subplots(2, len(RULES), figsize=(16, 7), sharey="row")
fig.suptitle("Train/Inference Gap — FC Layer (layer 2)", fontsize=14, fontweight="bold")

for col, rule in enumerate(RULES):
    cd = get_traj(rule, "CD", "fc")
    sm = get_traj(rule, "soft_margin_C")
    if cd is None:
        continue
    steps = np.arange(cd.shape[1])
    c = COLORS[rule]

    # Row 0: CD overlap
    ax = axes[0, col]
    ax.plot(steps, cd.mean(0), color=c, lw=2)
    ax.fill_between(steps, cd.min(0), cd.max(0), alpha=0.2, color=c)
    ax.axhline(1.0, ls="--", color="gray", lw=0.8)
    ax.set_title(RULE_LABELS[rule], fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("C↔D overlap" if col == 0 else "")
    ax.set_xlabel("weight updates")
    ax.grid(True, alpha=0.3)

    # Row 1: soft margin of C
    ax = axes[1, col]
    ax.plot(steps, sm.mean(0), color=c, lw=2)
    ax.fill_between(steps, sm.min(0), sm.max(0), alpha=0.2, color=c)
    ax.axhline(0.0, ls="--", color="gray", lw=0.8)
    ax.set_ylabel("soft margin(C)" if col == 0 else "")
    ax.set_xlabel("weight updates")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../figures/single_pattern/gap_figure1_CD_margin.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 2 — Full quadrilateral: all 6 pairwise overlaps per rule

Shows the complete geometry of A, B, C, D at the FC layer.

In [ ]:
PAIRS = ["AB", "BC", "CD", "AD", "AC", "BD"]
PAIR_STYLES = {"CD": dict(lw=2.5, ls="-"),   # highlight the gap metric
               "AB": dict(lw=1.2, ls="--"),
               "BC": dict(lw=1.2, ls="-."),
               "AD": dict(lw=1.2, ls=":"),
               "AC": dict(lw=1.2, ls=(0, (3, 1, 1, 1))),
               "BD": dict(lw=1.2, ls=(0, (5, 2)))}
PAIR_COLORS = {"CD": "black", "AB": "#E53935", "BC": "#43A047",
               "AD": "#FB8C00", "AC": "#8E24AA", "BD": "#00ACC1"}

fig, axes = plt.subplots(1, len(RULES), figsize=(18, 4), sharey=True)
fig.suptitle("Full Quadrilateral — FC Layer (layer 2)", fontsize=13, fontweight="bold")

for col, rule in enumerate(RULES):
    ax = axes[col]
    for pair in PAIRS:
        traj = get_traj(rule, pair, "fc")
        if traj is None:
            continue
        steps = np.arange(traj.shape[1])
        style = PAIR_STYLES[pair]
        ax.plot(steps, traj.mean(0), color=PAIR_COLORS[pair],
                label=pair, **style)
        ax.fill_between(steps, traj.min(0), traj.max(0),
                        alpha=0.08, color=PAIR_COLORS[pair])
    ax.axhline(1.0, ls="--", color="lightgray", lw=0.7)
    ax.axhline(0.0, ls="--", color="lightgray", lw=0.7)
    ax.set_title(RULE_LABELS[rule], fontsize=9)
    ax.set_ylim(-0.1, 1.1)
    ax.set_xlabel("weight updates")
    ax.grid(True, alpha=0.2)
    if col == 0:
        ax.set_ylabel("overlap m = mean(s_X · s_Y)")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=6, fontsize=9,
           bbox_to_anchor=(0.5, -0.05))
plt.tight_layout()
plt.savefig("../figures/single_pattern/gap_figure2_quadrilateral.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 3 — CD overlap comparison: FC vs Conv layer

Checks whether the gap story is consistent across layers.

In [ ]:
fig, axes = plt.subplots(1, len(RULES), figsize=(16, 4), sharey=True)
fig.suptitle("CD Overlap: FC vs Conv Layer", fontsize=13, fontweight="bold")

for col, rule in enumerate(RULES):
    ax = axes[col]
    for layer, ls, label in [("fc", "-", "FC (layer 2)"), ("conv", "--", "Conv (layer 1)")]:
        traj = get_traj(rule, "CD", layer)
        if traj is None:
            continue
        steps = np.arange(traj.shape[1])
        c = COLORS[rule]
        ax.plot(steps, traj.mean(0), color=c, ls=ls, lw=2, label=label)
        ax.fill_between(steps, traj.min(0), traj.max(0), alpha=0.15, color=c)
    ax.axhline(1.0, ls=":", color="gray", lw=0.8)
    ax.set_title(RULE_LABELS[rule], fontsize=9)
    ax.set_ylim(0.0, 1.05)
    ax.set_xlabel("weight updates")
    ax.grid(True, alpha=0.3)
    if col == 0:
        ax.set_ylabel("C↔D overlap")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, fontsize=9,
           bbox_to_anchor=(0.5, -0.08))
plt.tight_layout()
plt.savefig("../figures/single_pattern/gap_figure3_CD_layers.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 4 — Summary: final CD overlap and margin (bar chart)

Converged values (mean over last 10 steps, averaged across seeds).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle("Converged Values (mean over last 10 steps × all seeds)", fontsize=12, fontweight="bold")

metrics = [
    ("CD_fc",     "C↔D overlap (FC)",     "fc",   "CD"),
    ("margin_C",  "Soft margin of C",      None,   "soft_margin_C"),
    ("acc_D",     "Inference accuracy D",  None,   "accuracy_D"),
]

for ax, (_, ylabel, layer, key) in zip(axes, metrics):
    vals, errs, colors = [], [], []
    for rule in RULES:
        traj = get_traj(rule, key, layer) if layer else get_traj(rule, key)
        if traj is None:
            vals.append(0); errs.append(0)
        else:
            tail = traj[:, -10:].mean(1)   # mean over last 10 steps per seed
            vals.append(tail.mean())
            errs.append(tail.std())
        colors.append(COLORS[rule])

    short_labels = ["current", "approx", "EP", "long\nwarmup"]
    bars = ax.bar(short_labels, vals, color=colors, alpha=0.85, yerr=errs, capsize=4)
    ax.set_ylabel(ylabel)
    ax.grid(True, axis="y", alpha=0.3)
    ax.axhline(0, color="gray", lw=0.7, ls="--")

plt.tight_layout()
plt.savefig("../figures/single_pattern/gap_figure4_summary_bars.png", dpi=150, bbox_inches="tight")
plt.show()

# Print table
print(f"\n{'Rule':<15} {'CD_fc':>8} {'margin_C':>10} {'acc_D':>8}")
print("-" * 43)
for rule in RULES:
    cd   = get_traj(rule, "CD", "fc")[:, -10:].mean()
    sm   = get_traj(rule, "soft_margin_C")[:, -10:].mean()
    acc  = get_traj(rule, "accuracy_D")[:, -10:].mean()
    print(f"{rule:<15} {cd:>8.3f} {sm:>10.3f} {acc:>8.3f}")

## Figure 5 — EP α sweep: finding the right de-stabilization strength

Compares EP rule with different α values in `backward(C) - α·backward(B)`.
α=0 reduces to pure approximate rule; α=1 is full contrastive (broken).

In [ ]:
import glob

ALPHAS = [0.1, 0.3, 0.5, 0.7, 1.0]
ALPHA_COLORS = plt.cm.plasma(np.linspace(0.1, 0.9, len(ALPHAS)))

def get_ep_traj(alpha, key, layer="fc"):
    path = f"../results/single_pattern/gap_ep_alpha{alpha}.json"
    d = json.loads(Path(path).read_text())
    col = f"overlaps_{layer}"
    if key in ("soft_margin_C", "accuracy_D"):
        arrs = [[step[key] for step in r["trajectory"]] for r in d["results"]]
    else:
        arrs = [[step[col][key] for step in r["trajectory"]] for r in d["results"]]
    return np.array(arrs)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("EP Rule: α sweep  (update = backward(C) − α·backward(B))", fontsize=13, fontweight="bold")

for i, (ax, (key, layer, ylabel)) in enumerate(zip(axes, [
    ("CD",            "fc",  "C↔D overlap (FC)"),
    ("soft_margin_C", None,  "Soft margin of C"),
    ("accuracy_D",    None,  "Inference accuracy D"),
])):
    for alpha, color in zip(ALPHAS, ALPHA_COLORS):
        traj = get_ep_traj(alpha, key, layer) if layer else get_ep_traj(alpha, key)
        steps = np.arange(traj.shape[1])
        ax.plot(steps, traj.mean(0), color=color, lw=2, label=f"α={alpha}")
        ax.fill_between(steps, traj.min(0), traj.max(0), alpha=0.1, color=color)
    ax.axhline(1.0 if key == "CD" else 0.0, ls="--", color="gray", lw=0.8)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("weight updates")
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("../figures/single_pattern/gap_figure5_ep_alpha_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

# Summary table
print(f"\n{'α':<8} {'CD_fc':>8} {'margin_C':>10} {'acc_D':>8}")
print("-" * 36)
for alpha in ALPHAS:
    cd  = get_ep_traj(alpha, "CD", "fc")[:, -10:].mean()
    sm  = get_ep_traj(alpha, "soft_margin_C")[:, -10:].mean()
    acc = get_ep_traj(alpha, "accuracy_D")[:, -10:].mean()
    print(f"{alpha:<8} {cd:>8.3f} {sm:>10.3f} {acc:>8.3f}")

## Figure 6 — Original vs tuned hyperparams

Compares the default config (threshold=1.7, lr=0.05) against the best hyperparams found by the forgetting crossing-point criterion (Point 4).

Each rule uses its own tuned (threshold, lr) from `forgetting_multi_rule.py`:
- **current**: threshold=1.9, lr=0.01
- **approximate**: threshold=1.9, lr=0.1
- **long_warmup**: threshold=1.9, lr=0.01, n_warmup_long=10
- **ep**: threshold=1.9, lr=0.01, α=0.3

**Note on EP baseline:** `gap_results.json` ran EP with α=1.0 (the broken version that never converges). For a fair comparison, the EP "original" bar uses `gap_ep_alpha0.3.json` (thr=1.7, lr=0.05, α=0.3) — same default hyperparams but the correct α.

In [ ]:
TUNED_CONFIGS = {
    "current":     {"threshold": 1.9, "lr": 0.01},
    "approximate": {"threshold": 1.9, "lr": 0.1},
    "long_warmup": {"threshold": 1.9, "lr": 0.01, "n_warmup_long": 10},
    "ep":          {"threshold": 1.9, "lr": 0.01, "ep_alpha": 0.3},
}

# For EP, the original baseline is alpha=0.3 with default hyperparams.
# gap_results.json used alpha=1.0 (broken) for EP, so we use gap_ep_alpha0.3.json instead.
EP_ORIG_PATH = "../results/single_pattern/gap_ep_alpha0.3.json"

def get_traj_from_file(path, rule, key, layer="fc"):
    d = json.loads(Path(path).read_text())
    runs = [r for r in d["results"] if r["rule"] == rule]
    col = f"overlaps_{layer}"
    if key in ("soft_margin_C", "accuracy_D"):
        arrs = [[s[key] for s in r["trajectory"]] for r in runs]
    else:
        arrs = [[s[col][key] for s in r["trajectory"]] for r in runs]
    return np.array(arrs)

def get_orig_traj(rule, key, layer="fc"):
    """Original baseline: EP uses gap_ep_alpha0.3.json, others use gap_results.json."""
    if rule == "ep":
        return get_traj_from_file(EP_ORIG_PATH, "ep", key, layer)
    return get_traj(rule, key, layer) if layer else get_traj(rule, key)

# --- Figure 6a: CD trajectories original vs tuned per rule ---
fig, axes = plt.subplots(1, len(RULES), figsize=(16, 4), sharey=True)
fig.suptitle("CD_fc trajectory: original (solid) vs tuned (dashed)",
             fontsize=13, fontweight="bold")

for col, rule in enumerate(RULES):
    ax = axes[col]
    c = COLORS[rule]
    cfg_t = TUNED_CONFIGS[rule]

    # original
    cd_o = get_orig_traj(rule, "CD", "fc")
    steps = np.arange(cd_o.shape[1])
    orig_label = "original α=0.3 (thr=1.7, lr=0.05)" if rule == "ep" else "original (thr=1.7, lr=0.05)"
    ax.plot(steps, cd_o.mean(0), color=c, lw=2, ls="-", label=orig_label)
    ax.fill_between(steps, cd_o.min(0), cd_o.max(0), alpha=0.15, color=c)

    # tuned
    tuned_path = f"../results/single_pattern/gap_tuned_{rule}.json"
    cd_t = get_traj_from_file(tuned_path, rule, "CD", "fc")
    steps_t = np.arange(cd_t.shape[1])
    ax.plot(steps_t, cd_t.mean(0), color=c, lw=2, ls="--",
            label=f"tuned (thr={cfg_t['threshold']}, lr={cfg_t['lr']})")
    ax.fill_between(steps_t, cd_t.min(0), cd_t.max(0), alpha=0.1, color=c)

    ax.axhline(1.0, ls=":", color="gray", lw=0.8)
    ax.set_title(RULE_LABELS[rule], fontsize=9)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel("weight updates")
    ax.legend(fontsize=7, loc="lower right")
    ax.grid(True, alpha=0.3)
    if col == 0:
        ax.set_ylabel("C↔D overlap (FC)")

plt.tight_layout()
plt.savefig("../figures/single_pattern/gap_figure6a_orig_vs_tuned_CD.png",
            dpi=150, bbox_inches="tight")
plt.show()

# --- Figure 6b: side-by-side bar chart of final metrics ---
metrics_cfg = [
    ("CD",            "fc",  "C↔D overlap (FC)"),
    ("soft_margin_C", None,  "Soft margin of C"),
    ("accuracy_D",    None,  "Inference accuracy D"),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("Converged metrics: original vs tuned (last 10 steps × seeds)",
             fontsize=12, fontweight="bold")

short_labels = ["current", "approx", "EP", "long\nwarmup"]
x = np.arange(len(RULES))
width = 0.35

for ax, (key, layer, ylabel) in zip(axes, metrics_cfg):
    orig_vals, orig_errs = [], []
    tuned_vals, tuned_errs = [], []
    for rule in RULES:
        # original — EP uses alpha=0.3 file
        t = get_orig_traj(rule, key, layer)
        tail = t[:, -10:].mean(1)
        orig_vals.append(tail.mean()); orig_errs.append(tail.std())
        # tuned
        t2 = get_traj_from_file(
            f"../results/single_pattern/gap_tuned_{rule}.json", rule, key, layer
        ) if layer else get_traj_from_file(
            f"../results/single_pattern/gap_tuned_{rule}.json", rule, key
        )
        tail2 = t2[:, -10:].mean(1)
        tuned_vals.append(tail2.mean()); tuned_errs.append(tail2.std())

    rule_colors = [COLORS[r] for r in RULES]
    ax.bar(x - width/2, orig_vals, width, yerr=orig_errs,
           color=rule_colors, alpha=0.85, capsize=3, label="original")
    ax.bar(x + width/2, tuned_vals, width, yerr=tuned_errs,
           color=rule_colors, alpha=0.4, capsize=3,
           hatch="//", label="tuned")
    ax.set_xticks(x); ax.set_xticklabels(short_labels)
    ax.set_ylabel(ylabel)
    ax.axhline(0, color="gray", lw=0.7, ls="--")
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("../figures/single_pattern/gap_figure6b_orig_vs_tuned_bars.png",
            dpi=150, bbox_inches="tight")
plt.show()

# Summary table
print(f"\n{'Rule':<14} {'CD orig':>9} {'CD tuned':>9}  "
      f"{'margin orig':>12} {'margin tuned':>12}  "
      f"{'acc orig':>9} {'acc tuned':>9}")
print("-" * 80)
for rule in RULES:
    cd_o  = get_orig_traj(rule, "CD", "fc")[:, -10:].mean()
    sm_o  = get_orig_traj(rule, "soft_margin_C")[:, -10:].mean()
    ac_o  = get_orig_traj(rule, "accuracy_D")[:, -10:].mean()
    p = f"../results/single_pattern/gap_tuned_{rule}.json"
    cd_t  = get_traj_from_file(p, rule, "CD", "fc")[:, -10:].mean()
    sm_t  = get_traj_from_file(p, rule, "soft_margin_C")[:, -10:].mean()
    ac_t  = get_traj_from_file(p, rule, "accuracy_D")[:, -10:].mean()
    print(f"{rule:<14} {cd_o:>9.3f} {cd_t:>9.3f}  "
          f"{sm_o:>12.3f} {sm_t:>12.3f}  {ac_o:>9.3f} {ac_t:>9.3f}")